1.  **Regularization**:

    - Use the `diabetes` dataset from `sklearn.datasets`.
    - Compare the performance (Mean Squared Error) of `LinearRegression`, `Ridge`, and `Lasso` models.
    - Tune the `alpha` parameter for `Ridge` and `Lasso` using `GridSearchCV` with cross-validation to find the optimal regularization strength.

In [99]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline

# Load the diabetes dataset
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target
# print(X)
# print(X.isna().sum())

# Preprocessing for numerical data
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Bundle preprocessing for numerical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, diabetes.feature_names)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

linear_reg_model = LinearRegression()
linear_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('model', linear_reg_model)
                          ])
linear_pipeline.fit(X_train, y_train)
# print(linear_pipeline.named_steps['preprocessor'].get_feature_names_out())

# Generate predictions on the test set
linear_preds = linear_pipeline.predict(X_test)

# Calculate Mean Squared Error (MSE) to quantify prediction accuracy (lower is better)
linear_mse = mean_squared_error(y_test, linear_preds)
print("Linear Regression Mean Squared Error:", linear_mse)

Linear Regression Mean Squared Error: 3424.2593342986925


In [100]:
# Initialize a Ridge regression model
ridge_reg_model_tune = Ridge()

# Define the hyperparameter grid
ridge_param_grid = {
    'alpha': [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
}

# Initialize GridSearchCV with the model, parameter grid, cross-validation strategy, and scoring metrics
# here we use MSE score as the metrics
ridge_grid_search = GridSearchCV(estimator=ridge_reg_model_tune, param_grid=ridge_param_grid, cv=5, n_jobs=-1, verbose=2, scoring=('neg_mean_squared_error'))

# Create and evaluate the pipeline
ridge_pipeline_tune = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('grid_search', ridge_grid_search)
                          ])
ridge_pipeline_tune.fit(X_train, y_train)

# check best params
best_params = ridge_grid_search.best_params_
best_score = ridge_grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Score: {best_score:.4f}")

Fitting 5 folds for each of 6 candidates, totalling 30 fits


[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.4; total time=   0.0s
[CV] END ..........................................alpha=0.4; total time=   0.0s
[CV] END ..........................................alpha=0.4; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.4; total time=   0.0s
[CV] END ...................

In [101]:
ridge_grid_search.cv_results_

{'mean_fit_time': array([0.00219278, 0.00160985, 0.00136852, 0.00117478, 0.00107293,
        0.00114574]),
 'std_fit_time': array([0.00061387, 0.00067017, 0.00039833, 0.00031395, 0.00027763,
        0.00047887]),
 'mean_score_time': array([0.00076232, 0.00074611, 0.00074563, 0.00051212, 0.0005146 ,
        0.00044255]),
 'std_score_time': array([0.00035511, 0.00032673, 0.00019172, 0.00020286, 0.0001811 ,
        0.00014254]),
 'param_alpha': masked_array(data=[0.1, 0.2, 0.4, 0.6, 0.8, 1.0],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'alpha': 0.1},
  {'alpha': 0.2},
  {'alpha': 0.4},
  {'alpha': 0.6},
  {'alpha': 0.8},
  {'alpha': 1.0}],
 'split0_test_score': array([-2964.98125508, -2965.31670781, -2965.9823747 , -2966.61742653,
        -2967.20575873, -2967.73988625]),
 'split1_test_score': array([-2655.38697722, -2654.25953748, -2652.2577795 , -2650.53120186,
        -2649.02237567, -2647.68830501]),


In [102]:
# Initialize a Ridge regression model with best param
ridge_reg_model = Ridge(alpha=1.0)

ridge_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('model', ridge_reg_model)
                          ])
ridge_pipeline.fit(X_train, y_train)
ridge_preds = ridge_pipeline.predict(X_test)

# Calculate the Mean Squared Error (MSE)
ridge_mse = mean_squared_error(y_test, ridge_preds)
print(f'Ridge MSE: {ridge_mse}')


Ridge MSE: 3430.10676248457


In [103]:
# Initialize a Lasso regression model
lasso_reg_model_tune = Lasso()

# Define the hyperparameter grid
lasso_param_grid = {
    'alpha': [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
}

# Initialize GridSearchCV with the model, parameter grid, cross-validation strategy, and scoring metrics
# here we use MSE score as the metrics
lasso_grid_search = GridSearchCV(estimator=lasso_reg_model_tune, param_grid=lasso_param_grid, cv=5, n_jobs=-1, verbose=2, scoring=('neg_mean_squared_error'))

# Create and evaluate the pipeline
lasso_pipeline_tune = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('grid_search', lasso_grid_search)
                          ])
lasso_pipeline_tune.fit(X_train, y_train)

# check best params
best_params = lasso_grid_search.best_params_
best_score = lasso_grid_search.best_score_

print(f"Best Parameters: {best_params}")
print(f"Best Score: {best_score:.4f}")

Fitting 5 folds for each of 6 candidates, totalling 30 fits
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.1; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.2; total time=   0.0s
[CV] END ..........................................alpha=0.4; total time=   0.0s
[CV] END ..........................................alpha=0.6; total time=   0.0s
[CV] END ........................................

In [104]:
lasso_grid_search.cv_results_

{'mean_fit_time': array([0.00175285, 0.0017478 , 0.00153327, 0.00091896, 0.00113988,
        0.00090156]),
 'std_fit_time': array([6.98963970e-04, 1.14158246e-03, 4.18172078e-04, 7.61530887e-05,
        3.02268237e-04, 1.29087719e-04]),
 'mean_score_time': array([0.00052748, 0.00040359, 0.00102544, 0.000348  , 0.00048566,
        0.00034475]),
 'std_score_time': array([1.87000834e-04, 4.13309180e-05, 8.97250874e-04, 2.78378987e-05,
        2.03461839e-04, 2.54637293e-05]),
 'param_alpha': masked_array(data=[0.1, 0.2, 0.4, 0.6, 0.8, 1.0],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'alpha': 0.1},
  {'alpha': 0.2},
  {'alpha': 0.4},
  {'alpha': 0.6},
  {'alpha': 0.8},
  {'alpha': 1.0}],
 'split0_test_score': array([-2973.20974072, -2980.04253923, -2979.55485863, -2980.18058796,
        -2977.28224238, -2975.08400769]),
 'split1_test_score': array([-2647.77310833, -2641.37577522, -2636.44636628, -2636.9512

In [105]:
# Initialize a Lasso regression model with best param
lasso_reg_model = Lasso(alpha=0.4)

lasso_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('model', lasso_reg_model)
                          ])
lasso_pipeline.fit(X_train, y_train)

# Generate predictions on the test set using the trained Lasso model
lasso_preds = lasso_pipeline.predict(X_test)

# Calculate the Mean Squared Error (MSE)
lasso_mse = mean_squared_error(y_test, lasso_preds)
print(f'Lasso MSE: {lasso_mse}')


Lasso MSE: 3436.332562837374


In [106]:
print(f"LinearRegression MSE:", linear_mse, "\nRidge MSE:", ridge_mse, "\nLasso MSE:", lasso_mse)

LinearRegression MSE: 3424.2593342986925 
Ridge MSE: 3430.10676248457 
Lasso MSE: 3436.332562837374


2.  **Ensemble Methods**:

    - Use the `breast_cancer` dataset from `sklearn.datasets`.
    - Compare the performance (F1 Score and AUC) of `DecisionTreeClassifier`, `RandomForestClassifier`, and `GradientBoostingClassifier`.
    - Tune the hyperparameters of each classifier using `GridSearchCV` with cross-validation.

In [107]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, auc, roc_curve

# Load the breast cancer dataset
breast_cancer = load_breast_cancer(as_frame=True)
X = breast_cancer.data
y = breast_cancer.target
# print(X)
# print(X.isna().sum())

# Preprocessing for numerical data
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Bundle preprocessing for numerical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, breast_cancer.feature_names)
    ])

decision_tree_model_tune = DecisionTreeClassifier(criterion='gini', random_state=0)
decision_tree_param_grid = {
    'max_depth': [None, 2, 4, 6, 8, 10],
    'min_samples_split': [2, 4, 6, 8],
    'min_samples_leaf': [2, 4, 6, 8],
}

# Use f1 and auc score as the metrics
decision_tree_grid_search = GridSearchCV(estimator=decision_tree_model_tune, param_grid=decision_tree_param_grid, cv=5, n_jobs=-1, verbose=2, scoring=('f1','roc_auc'), refit=False)

decision_tree_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('grid_search', decision_tree_grid_search)
                          ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
decision_tree_pipeline.fit(X_train, y_train)


Fitting 5 folds for each of 96 candidates, totalling 480 fits
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.0s[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.0s

[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=4; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=4; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=2; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=4; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=4; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=6; total time=   0.0s
[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=6; total time=   0.0s
[CV

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  array(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'tex...
       'worst compactness', 'worst concavity', 'worst concave points',
       'worst symmetry', 'worst fractal dimension'], dtype='<U23'))])),
                ('grid_search',
                 GridSearchCV(cv=5,
                              estimator=DecisionTreeClassifier(random_state=0),
                              n_jobs=-1,
                              param_grid={'max_depth': [None, 2, 4, 6, 8, 10],
                                          'min_samples_leaf': [2, 4, 6, 8],
                                          'min_samples_split': [2, 4, 6, 8]},
                              refit=False, scoring=('f1', 'roc_auc'),
                              verbose=2))])

In [108]:
# Inspect results
import pandas as pd
results_df = pd.DataFrame(decision_tree_grid_search.cv_results_)

# Example: Get best params by accuracy
best_idx_auc = results_df['mean_test_roc_auc'].idxmax()
best_params_auc = results_df.loc[best_idx_auc, 'params']

# Example: Get best params by F1
best_idx_f1 = results_df['mean_test_f1'].idxmax()
best_params_f1 = results_df.loc[best_idx_f1, 'params']

print("Best params by F1:", best_params_f1)
print("Best params by AUC:", best_params_auc)

Best params by F1: {'max_depth': None, 'min_samples_leaf': 8, 'min_samples_split': 2}
Best params by AUC: {'max_depth': None, 'min_samples_leaf': 8, 'min_samples_split': 2}


In [109]:
list(zip(decision_tree_grid_search.cv_results_['params'],decision_tree_grid_search.cv_results_['mean_test_f1'], decision_tree_grid_search.cv_results_['mean_test_roc_auc']))

[({'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2},
  0.9393730786245083,
  0.9265412748171368),
 ({'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 4},
  0.9393730786245083,
  0.9265412748171368),
 ({'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 6},
  0.9374567169077445,
  0.9266980146290491),
 ({'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 8},
  0.9381902456601308,
  0.9331765935214211),
 ({'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 2},
  0.9437628542207227,
  0.9464994775339604),
 ({'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 4},
  0.9437628542207227,
  0.9464994775339604),
 ({'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 6},
  0.9437628542207227,
  0.9464994775339604),
 ({'max_depth': None, 'min_samples_leaf': 4, 'min_samples_split': 8},
  0.9437628542207227,
  0.9464994775339604),
 ({'max_depth': None, 'min_samples_leaf': 6, 'min_samples_split': 2},
  0.939058

In [110]:
# Initialize a decision tree model with default params
decision_tree_model = DecisionTreeClassifier(criterion='gini', random_state=0)
decision_tree_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('model', decision_tree_model)
                          ])
decision_tree_pipeline.fit(X_train, y_train)
decision_tree_preds = decision_tree_pipeline.predict(X_test)

f1 = f1_score(y_test, decision_tree_preds)
fpr, tpr, thresholds = roc_curve(y_test, decision_tree_pipeline.predict_proba(X_test)[:,1])
roc_auc = auc(fpr, tpr)

print(f'F1 Score: {f1:.2f}')
print(f"AUC: {roc_auc:.2f}")


F1 Score: 0.92
AUC: 0.92


In [111]:
# Initialize a decision tree model with best params
decision_tree_model_final = DecisionTreeClassifier(criterion='gini', random_state=0, max_depth=None, min_samples_split=2, min_samples_leaf=8)
decision_tree_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('model', decision_tree_model_final)
                          ])
decision_tree_pipeline.fit(X_train, y_train)
decision_tree_preds = decision_tree_pipeline.predict(X_test)

f1_final = f1_score(y_test, decision_tree_preds)
fpr_final, tpr_final, thresholds_final = roc_curve(y_test, decision_tree_pipeline.predict_proba(X_test)[:,1])
roc_auc_final = auc(fpr_final, tpr_final)

print(f'F1 Score: {f1_final:.2f}')
print(f"AUC: {roc_auc_final:.2f}")


F1 Score: 0.95
AUC: 0.99


In [112]:
print(f"Decision Tree default:\nF1 Score: {f1:.2f}, AUC: {roc_auc:.2f}")
print(f"Decision Tree tuned:\nF1 Score: {f1_final:.2f}, AUC: {roc_auc_final:.2f}")

Decision Tree default:
F1 Score: 0.92, AUC: 0.92
Decision Tree tuned:
F1 Score: 0.95, AUC: 0.99


In [113]:
random_forest_model_tune = RandomForestClassifier()
random_forest_param_grid = {
    'n_estimators': [50, 100, 150, 200, 250, 300],
    'max_depth': [2, 4, 6, 8, 10, 12],
    'min_samples_split': [2, 4, 6, 8],
    'min_samples_leaf': [2, 4, 6, 8]
}

# Use f1 and auc score as the metrics
random_forest_grid_search = GridSearchCV(estimator=random_forest_model_tune, param_grid=random_forest_param_grid, cv=5, n_jobs=-1, verbose=2, scoring=('f1','roc_auc'), refit=False)

random_forest_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('grid_search', random_forest_grid_search)
                          ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
random_forest_pipeline.fit(X_train, y_train)

Fitting 5 folds for each of 576 candidates, totalling 2880 fits
[CV] END max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   0.2s
[CV] END max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   0.2s
[CV] END max_depth=2, min_samples_leaf=2,

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  array(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'tex...
       'worst compactness', 'worst concavity', 'worst concave points',
       'worst symmetry', 'worst fractal dimension'], dtype='<U23'))])),
                ('grid_search',
                 GridSearchCV(cv=5, estimator=RandomForestClassifier(),
                              n_jobs=-1,
                              param_grid={'max_depth': [2, 4, 6, 8, 10, 12],
                                          'min_samples_leaf': [2, 4, 6, 8],
                                          'min_samples_split': [2, 4, 6, 8],
                                          'n_estimators': [50, 100, 150, 200,
                                                           250, 300]},
                              refit=False, scoring=('f1', 'roc_auc'),
                              verbose=2))])

In [114]:
# Inspect results
import pandas as pd
results_df = pd.DataFrame(random_forest_grid_search.cv_results_)

# Example: Get best params by accuracy
best_idx_auc = results_df['mean_test_roc_auc'].idxmax()
best_params_auc = results_df.loc[best_idx_auc, 'params']

# Example: Get best params by F1
best_idx_f1 = results_df['mean_test_f1'].idxmax()
best_params_f1 = results_df.loc[best_idx_f1, 'params']

print("Best params by F1:", best_params_f1)
print("Best params by AUC:", best_params_auc)

Best params by F1: {'max_depth': 6, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 50}
Best params by AUC: {'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 6, 'n_estimators': 100}


In [115]:
list(zip(random_forest_grid_search.cv_results_['params'],random_forest_grid_search.cv_results_['mean_test_f1'], random_forest_grid_search.cv_results_['mean_test_roc_auc']))

[({'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 2,
   'n_estimators': 50},
  0.9546425526732432,
  0.9846917450365724),
 ({'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 2,
   'n_estimators': 100},
  0.9575969976091123,
  0.9853709508881924),
 ({'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 2,
   'n_estimators': 150},
  0.9544110275689223,
  0.985423197492163),
 ({'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 2,
   'n_estimators': 200},
  0.9562662530008318,
  0.9866248693834899),
 ({'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 2,
   'n_estimators': 250},
  0.9560627391009927,
  0.9854754440961336),
 ({'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 2,
   'n_estimators': 300},
  0.9578628916562939,
  0.9861024033437827),
 ({'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 4,
   'n_estimators': 50},
  0.9631590801443626,
  0.985997910135841),
 ({'max_de

In [ ]:
# Initialize a random forest model with default params
random_forest_model = RandomForestClassifier()

random_forest_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('model', random_forest_model)
                          ])
random_forest_pipeline.fit(X_train, y_train)
random_forest_preds = random_forest_pipeline.predict(X_test)

f1 = f1_score(y_test, random_forest_preds)
fpr, tpr, thresholds = roc_curve(y_test, random_forest_pipeline.predict_proba(X_test)[:,1])
roc_auc = auc(fpr, tpr)

print(f'F1 Score: {f1:.2f}')
print(f"AUC: {roc_auc:.2f}")


F1 Score: 0.97
AUC: 1.00


In [ ]:
# Initialize a random forest model with best params
random_forest_model_final= RandomForestClassifier(n_estimators=50, max_depth=6, min_samples_split=2, min_samples_leaf=2)

random_forest_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('model', random_forest_model_final)
                          ])
random_forest_pipeline.fit(X_train, y_train)
random_forest_preds = random_forest_pipeline.predict(X_test)

f1_final = f1_score(y_test, random_forest_preds)
fpr_final, tpr_final, thresholds_final = roc_curve(y_test, random_forest_pipeline.predict_proba(X_test)[:,1])
roc_auc_final = auc(fpr_final, tpr_final)

print(f'F1 Score: {f1_final:.2f}')
print(f"AUC: {roc_auc_final:.2f}")


F1 Score: 0.98
AUC: 1.00


In [118]:
print(f"Random Forest default:\nF1 Score: {f1:.2f}, AUC: {roc_auc:.2f}")
print(f"Random Forest tuned:\nF1 Score: {f1_final:.2f}, AUC: {roc_auc_final:.2f}")

Random Forest default:
F1 Score: 0.97, AUC: 1.00
Random Forest tuned:
F1 Score: 0.98, AUC: 1.00


In [121]:
gradient_boosting_model_tune = GradientBoostingClassifier()
gradient_boosting_param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [2, 4, 6, 8],
    'min_samples_split': [2, 4, 6, 8],
    'min_samples_leaf': [2, 4, 6, 8]
}

# Use f1 and auc score as the metrics
gradient_boosting_grid_search = GridSearchCV(estimator=gradient_boosting_model_tune, param_grid=gradient_boosting_param_grid, cv=5, n_jobs=-1, verbose=2, scoring=('f1','roc_auc'), refit=False)

gradient_boosting_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('grid_search', gradient_boosting_grid_search)
                          ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
gradient_boosting_pipeline.fit(X_train, y_train)

Fitting 5 folds for each of 1024 candidates, totalling 5120 fits
[CV] END learning_rate=0.01, max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.2s
[CV] END learning_rate=0.01, max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.2s
[CV] END learning_rate=0.01, max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.2s
[CV] END learning_rate=0.01, max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.2s
[CV] END learning_rate=0.01, max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=50; total time=   0.2s
[CV] END learning_rate=0.01, max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=   0.4s
[CV] END learning_rate=0.01, max_depth=2, min_samples_leaf=2, min_samples_split=2, n_estimators=100;

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  array(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'tex...
       'worst symmetry', 'worst fractal dimension'], dtype='<U23'))])),
                ('grid_search',
                 GridSearchCV(cv=5, estimator=GradientBoostingClassifier(),
                              n_jobs=-1,
                              param_grid={'learning_rate': [0.01, 0.05, 0.1,
                                                            0.2],
                                          'max_depth': [2, 4, 6, 8],
                                          'min_samples_leaf': [2, 4, 6, 8],
                                          'min_samples_split': [2, 4, 6, 8],
                                          'n_estimators': [50, 100, 150, 200]},
                              refit=False, scoring=('f1', 'roc_auc'),
                              verbose=2))])

In [122]:
# Inspect results
import pandas as pd
results_df = pd.DataFrame(gradient_boosting_grid_search.cv_results_)

# Example: Get best params by accuracy
best_idx_auc = results_df['mean_test_roc_auc'].idxmax()
best_params_auc = results_df.loc[best_idx_auc, 'params']

# Example: Get best params by F1
best_idx_f1 = results_df['mean_test_f1'].idxmax()
best_params_f1 = results_df.loc[best_idx_f1, 'params']

print("Best params by F1:", best_params_f1)
print("Best params by AUC:", best_params_auc)

Best params by F1: {'learning_rate': 0.2, 'max_depth': 6, 'min_samples_leaf': 8, 'min_samples_split': 2, 'n_estimators': 50}
Best params by AUC: {'learning_rate': 0.2, 'max_depth': 2, 'min_samples_leaf': 6, 'min_samples_split': 8, 'n_estimators': 200}


In [123]:
list(zip(gradient_boosting_grid_search.cv_results_['params'],gradient_boosting_grid_search.cv_results_['mean_test_f1'], gradient_boosting_grid_search.cv_results_['mean_test_roc_auc']))

[({'learning_rate': 0.01,
   'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 2,
   'n_estimators': 50},
  0.9411705147825927,
  0.9610240334378265),
 ({'learning_rate': 0.01,
   'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 2,
   'n_estimators': 100},
  0.9526340867830557,
  0.975496342737722),
 ({'learning_rate': 0.01,
   'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 2,
   'n_estimators': 150},
  0.9524570918025267,
  0.9845350052246605),
 ({'learning_rate': 0.01,
   'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 2,
   'n_estimators': 200},
  0.9524867234899125,
  0.9856844305120166),
 ({'learning_rate': 0.01,
   'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 4,
   'n_estimators': 50},
  0.9411705147825927,
  0.9609195402298851),
 ({'learning_rate': 0.01,
   'max_depth': 2,
   'min_samples_leaf': 2,
   'min_samples_split': 4,
   'n_estimators': 100},
  0.9526340867830557,
  0.97549634273772

In [133]:
# Initialize a gradient boosting model with default params
gradient_boosting_model = GradientBoostingClassifier(random_state=2)

gradient_boosting_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('model', gradient_boosting_model)
                          ])
gradient_boosting_pipeline.fit(X_train, y_train)
gradient_boosting_preds = gradient_boosting_pipeline.predict(X_test)

f1 = f1_score(y_test, gradient_boosting_preds)
fpr, tpr, thresholds = roc_curve(y_test, gradient_boosting_pipeline.predict_proba(X_test)[:,1])
roc_auc = auc(fpr, tpr)

print(f'F1 Score: {f1:.2f}')
print(f"AUC: {roc_auc:.2f}")


F1 Score: 0.97
AUC: 1.00


In [134]:
# Initialize a gradient boosting model with best params
gradient_boosting_model_final = GradientBoostingClassifier(n_estimators=50, learning_rate=0.2, max_depth=6, min_samples_split=2, min_samples_leaf=8)

gradient_boosting_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('model', gradient_boosting_model_final)
                          ])
gradient_boosting_pipeline.fit(X_train, y_train)
gradient_boosting_preds = gradient_boosting_pipeline.predict(X_test)

f1_final = f1_score(y_test, gradient_boosting_preds)
fpr_final, tpr_final, thresholds_final = roc_curve(y_test, gradient_boosting_pipeline.predict_proba(X_test)[:,1])
roc_auc_final = auc(fpr_final, tpr_final)

print(f'F1 Score: {f1_final:.2f}')
print(f"AUC: {roc_auc_final:.2f}")


F1 Score: 0.98
AUC: 1.00


In [135]:
print(f"Gradient Boosting default:\nF1 Score: {f1:.2f}, AUC: {roc_auc:.2f}")
print(f"Gradient Boosting tuned:\nF1 Score: {f1_final:.2f}, AUC: {roc_auc_final:.2f}")

Gradient Boosting default:
F1 Score: 0.97, AUC: 1.00
Gradient Boosting tuned:
F1 Score: 0.98, AUC: 1.00
